In [1]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

### Exercise 2

Construct an MLP containing a shared parameter layer and train it. During the training process, observe the model parameters and gradients of each layer.

In [2]:
torch.manual_seed(0)

# 256 examples, 4 input features
X = torch.randn(256, 4)
true_w = torch.tensor([[2.0], [-3.4], [1.5], [0.7]])
true_b = 0.5
y = X @ true_w + true_b + 0.1 * torch.randn(256, 1)

dataset = TensorDataset(X, y)
data_iter = DataLoader(dataset, batch_size=32, shuffle=True)

In [3]:
shared = nn.LazyLinear(8)

net = nn.Sequential(
    nn.LazyLinear(8), nn.ReLU(),
    shared, nn.ReLU(),
    shared, nn.ReLU(),
    nn.LazyLinear(1)
)

/Users/zouminghao/Desktop/d2l-notes-exercises/venv/lib/python3.11/site-packages/torch/nn/modules/lazy.py:180: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


In [4]:
net(X[:2])

tensor([[0.1246],
        [0.1259]], grad_fn=<AddmmBackward0>)

In [5]:
print(net[2] is net[4])
print(net[2].weight is net[4].weight)
print(net[2].bias is net[4].bias)

True
True
True


In [6]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(net.parameters(), lr=0.05)

In [7]:
num_epochs = 5

for epoch in range(num_epochs):
    for batch_idx, (X_batch, y_batch) in enumerate(data_iter):
        optimizer.zero_grad()
        
        y_hat = net(X_batch)
        loss = loss_fn(y_hat, y_batch)
        loss.backward()

        # Observe gradients before the parameter update
        if batch_idx == 0:  # Only print for the first batch of each epoch
            print(f"\nEpoch {epoch+1}")
            print(f"Loss: {loss.item():.6f}")
            
            print("Shared layer identity checks:")
            print("net[2] is net[4]:", net[2] is net[4])
            print("net[2].weight is net[4].weight:", net[2].weight is net[4].weight)
            print("net[2].bias is net[4].bias:", net[2].bias is net[4].bias)
            
            print("\nShared weight first row:")
            print("net[2].weight.data[0] =", net[2].weight.data[0])
            print("net[4].weight.data[0] =", net[4].weight.data[0])

            print("\nShared weight gradient first row:")
            print("net[2].weight.grad[0] =", net[2].weight.grad[0])
            print("net[4].weight.grad[0] =", net[4].weight.grad[0])

            print("\nAre gradients equal?")
            print(torch.equal(net[2].weight.grad, net[4].weight.grad))

        optimizer.step()


Epoch 1
Loss: 26.121700
Shared layer identity checks:
net[2] is net[4]: True
net[2].weight is net[4].weight: True
net[2].bias is net[4].bias: True

Shared weight first row:
net[2].weight.data[0] = tensor([-0.0220,  0.0632, -0.0538, -0.2471, -0.3304,  0.2325, -0.0421, -0.1313])
net[4].weight.data[0] = tensor([-0.0220,  0.0632, -0.0538, -0.2471, -0.3304,  0.2325, -0.0421, -0.1313])

Shared weight gradient first row:
net[2].weight.grad[0] = tensor([-0.0163, -0.0241,  0.0538, -0.0869,  0.1571,  0.2635,  0.1123,  0.0791])
net[4].weight.grad[0] = tensor([-0.0163, -0.0241,  0.0538, -0.0869,  0.1571,  0.2635,  0.1123,  0.0791])

Are gradients equal?
True

Epoch 2
Loss: 20.006248
Shared layer identity checks:
net[2] is net[4]: True
net[2].weight is net[4].weight: True
net[2].bias is net[4].bias: True

Shared weight first row:
net[2].weight.data[0] = tensor([-0.0198,  0.0752, -0.0725, -0.2036, -0.3819,  0.1760, -0.0855, -0.1314])
net[4].weight.data[0] = tensor([-0.0198,  0.0752, -0.0725, -0.203